<img src="https://drive.google.com/uc?export=view&id=1wYSMgJtARFdvTt5g7E20mE4NmwUFUuog" width="200">

## Agno Skills: Build Reusable Knowledge Packages for Your AI Agents
*BuildFastWithAI — Hands-On Workshop*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

---

### 🎯 What You'll Learn
- What Agno Skills are and why they beat monolithic system prompts
- How to write a `SKILL.md` file (the right format)
- How to add optional `scripts/` and `references/` to a skill
- How to load skills into an Agno agent using `LocalSkills`
- How to load multiple skill sets at once
- How the agent automatically uses skill tools at runtime

**Time:** ~35 minutes | **Level:** Intermediate | **Framework:** [Agno](https://agno.com)

---

## 📦 Section 1: Setup & Installation

In [ ]:
!pip install agno openai -q

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("🔑 Enter your OpenAI API key: ")
print("✅ API key set!")

---

## 🧠 Section 2: What Are Agno Skills?

### The Problem with Monolithic System Prompts

Most developers do this:

```python
# ❌ Bad: Everything crammed into one prompt
agent = Agent(
    instructions="""
    You are an expert code reviewer AND a Twitter content writer AND a SQL analyst AND...
    [500 more lines]
    """
)
```

The agent burns tokens on ALL instructions for EVERY request — even irrelevant ones.

### The Agno Skills Solution

**Skills** are self-contained knowledge packages organized as folders:

```
my-skill/
├── SKILL.md           ← Required: instructions with YAML frontmatter
├── scripts/           ← Optional: Python/shell scripts the agent can run
│   └── helper.py
└── references/        ← Optional: docs, guides, cheatsheets
    └── guide.md
```

### Progressive Discovery (Why This Is Smart)

| Step | What Happens |
|------|-------------|
| 1️⃣ **Browse** | Agent sees skill names + descriptions in system prompt |
| 2️⃣ **Load** | When task matches, agent calls `get_skill_instructions()` |
| 3️⃣ **Reference** | Agent fetches docs from `references/` only when needed |
| 4️⃣ **Execute** | Agent can run `scripts/` using `get_skill_script()` |

> Only what's needed is loaded. Context stays lean. Costs stay low.

---

## 📝 Section 3: Writing a SKILL.md File

The `SKILL.md` is the heart of every skill. It uses YAML frontmatter + Markdown.

### Required Fields

```yaml
---
name: skill-name          # lowercase, alphanumeric + hyphens, max 64 chars
description: Short desc   # shown to the agent in system prompt, max 1024 chars
---
```

### Optional Fields

```yaml
---
name: code-review
description: Reviews Python code for bugs, style, and best practices
license: Apache-2.0
metadata:
  version: "1.0.0"
  author: buildfastwithai
  tags: ["python", "code-quality"]
---
```

### Naming Rules
- ✅ `code-review` (valid)
- ✅ `twitter-writer` (valid)
- ❌ `Code_Review` (no uppercase, no underscores)
- ❌ `-code-review` (can't start with hyphen)
- ❌ `code--review` (no consecutive hyphens)

---

## 🔨 Section 4: Build Your First Skill — Code Review

Let's create a complete `code-review` skill from scratch, programmatically.

In [ ]:
import os
from pathlib import Path

# Create the skill directory structure
skill_root = Path("./skills")
code_review_skill = skill_root / "code-review"
scripts_dir = code_review_skill / "scripts"
references_dir = code_review_skill / "references"

for d in [code_review_skill, scripts_dir, references_dir]:
    d.mkdir(parents=True, exist_ok=True)

print("📁 Skill directory structure created:")
for p in sorted(skill_root.rglob("*")):
    print(f"  {p}")

In [ ]:
# ── Write SKILL.md ──
skill_md_content = """\
---
name: code-review
description: Reviews Python code for bugs, style issues, and best practices. Use when the user asks for code review, feedback, or wants to improve code quality.
license: MIT
metadata:
  version: "1.0.0"
  author: buildfastwithai
  tags: ["python", "code-quality", "review"]
---

# Code Review Skill

Use this skill when reviewing code for quality, style, and correctness.

## When to Use

- User asks for a code review or feedback
- User wants to improve code quality
- User needs help with refactoring
- User wants to check for bugs or security issues

## Review Process

1. **Analyze Structure**: Review overall code organization and readability
2. **Check Style**: Look for PEP 8 violations and naming conventions
3. **Identify Issues**: Find bugs, security issues, performance problems
4. **Suggest Improvements**: Provide actionable recommendations with examples

## Output Format

Always return:
- ✅ What's done well
- ⚠️  Issues found (with line references)
- 💡 Suggested improvements (with corrected code)
- 📊 Overall rating (1-10)

## Best Practices

- Focus on the most impactful issues first
- Explain the *why* behind each suggestion
- Provide corrected code examples, not just critique
- Use the style-guide reference for PEP 8 details if needed
"""

(code_review_skill / "SKILL.md").write_text(skill_md_content)
print("✅ SKILL.md written")

In [ ]:
# ── Write a Reference Doc ──
style_guide = """\
# Python Style Guide (PEP 8 Quick Reference)

## Naming Conventions
- **Variables & functions**: `snake_case`
- **Classes**: `PascalCase`
- **Constants**: `UPPER_SNAKE_CASE`
- **Private**: prefix with `_`

## Line Length
- Maximum **79 characters** per line
- Break long lines at logical points using `\\`

## Imports
Order: stdlib → third-party → local (alphabetized within each group)

## Spacing
- 2 blank lines between top-level functions/classes
- 1 blank line between methods inside a class
- No trailing whitespace

## Docstrings
- All public functions/classes need docstrings
- Use triple double-quotes: `"""This does X."""`
"""

(references_dir / "style-guide.md").write_text(style_guide)
print("✅ references/style-guide.md written")

In [ ]:
# ── Write an Optional Script ──
check_style_script = """\
#!/usr/bin/env python3
"""Check basic Python style issues and return a report."""

import sys

def check_style(code: str) -> dict:
    issues = []
    lines = code.split('\\n')

    for i, line in enumerate(lines, 1):
        if len(line) > 79:
            issues.append(f"Line {i}: exceeds 79 characters ({len(line)} chars)")
        if line.endswith(' '):
            issues.append(f"Line {i}: trailing whitespace")
        if '\\t' in line:
            issues.append(f"Line {i}: tab character (use spaces)")

    return {"issues": issues, "count": len(issues), "clean": len(issues) == 0}

if __name__ == "__main__":
    code = sys.stdin.read() if not sys.argv[1:] else open(sys.argv[1]).read()
    result = check_style(code)
    if result["clean"]:
        print("✅ No style issues found!")
    else:
        print(f"⚠️  {result['count']} issue(s) found:")
        for issue in result["issues"]:
            print(f"  - {issue}")
"""

(scripts_dir / "check_style.py").write_text(check_style_script)
print("✅ scripts/check_style.py written")

# Final directory tree
print("\n📁 Complete skill structure:")
for p in sorted(skill_root.rglob("*")):
    indent = "  " * (len(p.parts) - len(skill_root.parts))
    print(f"{indent}{p.name}")

---

## 🤖 Section 5: Loading the Skill into an Agent

The `Skills` class with `LocalSkills` loader is all you need.

In [ ]:
from pathlib import Path
from agno.agent import Agent
from agno.models.openai import OpenAIResponses
from agno.skills import Skills, LocalSkills

skills_dir = Path("./skills")

# Create the agent — skills are loaded automatically
agent = Agent(
    name="CodeAssistant",
    model=OpenAIResponses(id="gpt-4o"),
    skills=Skills(
        loaders=[LocalSkills(str(skills_dir))]   # loads all skills in the dir
    ),
    instructions=[
        "You are a helpful coding assistant with access to specialized skills.",
        "Use get_skill_instructions to load full guidance when the task requires it.",
    ],
    markdown=True,
)

print("🤖 Code Assistant ready with Skills!")

In [ ]:
# The agent automatically discovers and uses the code-review skill
agent.print_response(
    "Can you review this Python function?\n\n"
    "def calc(x,y):return x+y",
    stream=True,
)

### 🔍 What Tools Does the Agent Have?

When you add `skills=Skills(...)`, the agent automatically gets 3 built-in tools:

| Tool | What It Does |
|------|--------------|
| `get_skill_instructions(skill_name)` | Loads the full `SKILL.md` content |
| `get_skill_reference(skill_name, reference_path)` | Loads a file from `references/` |
| `get_skill_script(skill_name, script_path, execute, args, timeout)` | Reads or runs a file from `scripts/` |

You don't wire these up — the agent decides when to call them.

---

## 🗂️ Section 6: Loading Multiple Skills

Build a second skill and load both at once.

In [ ]:
# Create a second skill: twitter-writer
twitter_skill_dir = skill_root / "twitter-writer"
twitter_skill_dir.mkdir(parents=True, exist_ok=True)

twitter_skill_md = """\
---
name: twitter-writer
description: Writes high-engagement tweets and threads for the AI/developer audience on the BuildFastWithAI account. Use when the user wants to create Twitter content.
license: MIT
metadata:
  version: "1.0.0"
  author: buildfastwithai
  tags: ["twitter", "content", "social-media"]
---

# Twitter Writer Skill

Use this skill when creating Twitter content for the @BuildFastWithAI account.

## When to Use

- User wants a tweet or thread about an AI topic
- User wants to announce a new project, notebook, or tutorial
- User wants to explain a technical concept for a dev audience

## Audience Profile

- AI engineers and developers (intermediate to advanced)
- They value: code snippets, concrete examples, honest opinions
- They avoid: hype without substance, vague statements

## Writing Rules

- Hook tweet: max 280 chars, strong opening line
- Threads: 6-9 tweets, each self-contained
- Use emojis sparingly (1-2 per tweet max)
- Include a code snippet in at least one tweet when technical
- End thread with a CTA (link to notebook, drop a 🔥, etc.)
- Hashtags: max 3, always include #BuildFastWithAI

## Format

Return:
1. Single hook tweet (standalone)
2. Full thread (numbered tweets)
"""

(twitter_skill_dir / "SKILL.md").write_text(twitter_skill_md)
print("✅ twitter-writer skill created")

In [ ]:
from pathlib import Path
from agno.agent import Agent
from agno.models.openai import OpenAIResponses
from agno.skills import Skills, LocalSkills

# Two separate skill paths
code_review_path = Path("./skills/code-review")
twitter_writer_path = Path("./skills/twitter-writer")

# Load multiple skills — each loader can point to a single skill or whole directory
multi_skill_agent = Agent(
    name="ContentDevAgent",
    model=OpenAIResponses(id="gpt-4o"),
    skills=Skills(
        loaders=[
            LocalSkills(str(code_review_path)),
            LocalSkills(str(twitter_writer_path)),
        ]
    ),
    instructions=[
        "You are a developer content assistant with access to specialized skills.",
        "Always use get_skill_instructions before performing a specialized task.",
    ],
    markdown=True,
)

print("🤖 Multi-skill agent ready!")
print("   Skills loaded: code-review, twitter-writer")

In [ ]:
# Test with a Twitter content request
multi_skill_agent.print_response(
    "Write a tweet thread explaining Agno Skills for the @BuildFastWithAI audience.",
    stream=True,
)

---

## 🔄 Section 7: Advanced — Reloading & Error Handling

In [ ]:
from agno.skills import Skills, LocalSkills, SkillValidationError

# ── Reloading skills at runtime ──
# Useful if you update SKILL.md files without restarting
skills = Skills(loaders=[LocalSkills(str(Path("./skills")))])

# ... modify a SKILL.md on disk ...
# skills.reload()   # ← picks up new changes

print("✅ Skills can be reloaded at runtime with skills.reload()")

# ── Error handling ──
print("\n--- Validation Error Example ---")
try:
    bad_skills = Skills(loaders=[LocalSkills("/path/that/does/not/exist")])
except SkillValidationError as e:
    print(f"⚠️  Skill validation failed: {e}")
except Exception as e:
    print(f"ℹ️  Error (expected for demo): {type(e).__name__}: {e}")

---

## 🏋️ Section 8: Challenge — Build Your Own Skill

Pick one of these and build the full skill folder:

**Option A — SQL Analyst Skill**
- Instructions: explain how to read/write SQL queries
- Reference: common SQL patterns cheatsheet
- Script: a Python script that validates a SQL query string

**Option B — Git Workflow Skill**
- Instructions: explain commit conventions, PR templates, branching strategies
- Reference: conventional commits guide
- Script: a shell script to format a commit message

**Option C — Data Viz Skill**
- Instructions: when and how to choose chart types, color palettes, axis labels
- Reference: chart type selection guide
- Script: a Python script that generates a boilerplate matplotlib/seaborn chart

In [ ]:
# 🏋️ Your Challenge: Create your own skill here!
from pathlib import Path

MY_SKILL_NAME = "my-custom-skill"   # ← change this
_dir = Path("./skills") / MY_SKILL_NAME
_dir.mkdir(parents=True, exist_ok=True)
(_dir / "references").mkdir(exist_ok=True)
(_dir / "scripts").mkdir(exist_ok=True)

MY_SKILL_MD = f"""\
---
name: {MY_SKILL_NAME}
description: TODO — describe what this skill does
---

# My Custom Skill

## When to Use
- TODO

## Process
1. TODO
"""

(_dir / "SKILL.md").write_text(MY_SKILL_MD)
print(f"🛠️  Skill scaffold created at ./skills/{MY_SKILL_NAME}/SKILL.md")
print("   Now fill in the SKILL.md and load it into an agent!")

---

## 🎓 Summary

| Concept | Key Takeaway |
|---------|-------------|
| **SKILL.md** | YAML frontmatter + Markdown instructions — the heart of every skill |
| **scripts/** | Optional Python/shell scripts the agent can read or execute |
| **references/** | Optional docs loaded on demand — no wasted tokens |
| **LocalSkills** | Loads one skill folder or a directory of skills |
| **Skills** | Orchestrator that holds multiple loaders |
| **Progressive Loading** | Agent browses → loads → references → executes. Always lazy. |
| **Reusability** | One skill, used across every agent you build |

### 🔗 Resources
- [Agno Skills Documentation](https://docs.agno.com)
- [Agno GitHub Repository](https://github.com/agno-agi/agno)
- [Anthropic Agent Skills Spec](https://platform.claude.com/docs/en/agents-and-tools/agent-skills/overview)

---

**Built with ❤️ by [BuildFastWithAI](https://buildfastwithai.com)**  
Follow [@BuildFastWithAI](https://twitter.com/buildfastwithai) for more hands-on AI notebooks every week!